# S1 Demo — Data Profiling (Topic 1.2)
Raw Telco churn data → profile → clean → save. **Session 2 trains on the file this notebook saves.**
Runs identically on a laptop and in SageMaker Studio.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [ ]:
from course_utils import data
df = data.load_raw('telco_churn')
df.head()

## 1. First look — shapes and types
Spot the smell: `TotalCharges` is `object`, not a number.

In [ ]:
data.profile_frame(df)

## 2. The hidden blanks
`TotalCharges` contains spaces masquerading as values — coercion exposes them.

In [ ]:
import pandas as pd
tc = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('blank/broken TotalCharges:', tc.isna().sum())
df[tc.isna()][['customerID', 'tenure', 'TotalCharges']].head()

11 rows, all `tenure == 0` (brand-new customers). Decision: **drop** — they carry no churn signal yet. (Imputing 0 is defensible too; the point is to *decide and document*.)

## 3. Class balance — is 73% accuracy good?

In [ ]:
counts = df['Churn'].value_counts(normalize=True)
print(counts.round(3))
counts.plot.bar(title='Churn class balance');

## 4. Distributions & outliers

In [ ]:
df_num = df.assign(TotalCharges=tc)
df_num[['tenure', 'MonthlyCharges', 'TotalCharges']].plot.box(subplots=True, figsize=(10, 3));

Nothing pathological — *finding nothing is also a result.*

## 5. Clean & save → the pipeline starts here

In [ ]:
clean = data.clean_telco(df)
print(clean.shape, '| churn rate:', clean['Churn'].mean().round(3))
data.save_processed(clean, 'telco_clean')